# Full Evaluation Metrics — Real Data, GroupKFold(5)

**2026-09-24 rewrite (real data, supersedes the synthetic-data run below the fold).** This notebook
now runs on the real assembled dataset. Two honest evaluations, per `real_dataset_setup.md` §2/Phase
1.9:

1. **Eval 1 — clinical + MRI only, on all ~385 OASIS subjects** (`data/oasis_pool.csv`), subject-grouped
   CV. Full strength, no speech bottleneck.
2. **Eval 2 — fused (clinical+MRI+speech), on the windowed matched set**
   (`data/multimodal_dementia_dataset.csv`, 233 rows), `GroupKFold(5)` on the composite
   `speaker_id|oasis_subject_id` group — every window from one speaker, and every OASIS subject
   matched to it, stays on one side of every split.

Also: per-modality baselines within Eval 2, a **speech-only bootstrap CI** (only ~53 distinct speaker
groups feed that branch), **sex-stratified** rows (disclosed, not hidden, despite the small female
subgroup), a **sensitivity table** (matching richness / `R_max`, raw-vs-clean audio, noise on/off),
and the Phase 1.9 sanity checks (fused >= best single modality; fused not saturating while speech-only
is near chance; windowed fused not far above the strict 1:1 comparison build).

`SIGMA_FRAC = 0` for the headline (real data is not trivially separable — see the leak probe in
`scripts/verify_real_dataset.py`); noise is kept as one axis of the sensitivity table, not the default.

**Positive class = Demented (`Label == 1`)** throughout. Precision, recall and F1 are reported for that
class; specificity is the true-negative rate on Nondemented.

---
### Original synthetic-data note (superseded, kept for history)
The original version of this notebook re-ran `multimodal_qsvm.ipynb`'s synthetic-data,
`StratifiedKFold`, `SIGMA_FRAC=1.5` experiment with a fuller metric set, and asserted a hardcoded
`EXPECTED_FOLD1` confusion matrix as a reproduction check. Both are gone: `StratifiedKFold` doesn't
belong on grouped, real, matched data, and a hardcoded confusion matrix belongs to a fixed random draw
that no longer exists. The reproduction check below instead recomputes the fused accuracy with this
notebook's own `GroupKFold(5)` pipeline and asserts it matches the freshly written
`results/multimodal_results.json` (i.e. "this notebook's numbers agree with `multimodal_qsvm.ipynb`'s
numbers", not "these numbers match some earlier synthetic run").

In [1]:
import json
import time
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityStatevectorKernel
from qiskit_machine_learning.algorithms import QSVC
import warnings
warnings.filterwarnings('ignore')

# --- Eval 2 input: the windowed, matched real dataset ---
df = pd.read_csv('data/multimodal_dementia_dataset.csv')
prov = pd.read_csv('data/multimodal_real_provenance.csv')
df = df.merge(prov[['Subject_ID', 'group_id', 'sex']], on='Subject_ID', how='left')
assert df['group_id'].isna().sum() == 0
y = df['Label'].values
groups = df['group_id'].values

GROUPS = {
    'clinical': ['MMSE', 'ASF', 'EDUC', 'SES'],   # CDR dropped -- re-injects the label (locked decision)
    'mri':      ['nWBV', 'eTIV'],
    'speech':   ['pause_rate', 'speech_rate', 'pitch_mean', 'jitter', 'shimmer']
                + [f'mfcc_{i}' for i in range(1, 14)],
}
ALL_FEATURES = sum(GROUPS.values(), [])
assert len(ALL_FEATURES) == 24, len(ALL_FEATURES)

X_raw = df[ALL_FEATURES].values.astype(float)
feature_std = df[ALL_FEATURES].std().values
COLS = {g: [ALL_FEATURES.index(c) for c in GROUPS[g]] for g in GROUPS}

SIGMA_FRAC = 0   # headline: no noise. Sensitivity table below also runs SIGMA_FRAC=1.5.
rng = np.random.default_rng(42)
X_noisy = X_raw + rng.normal(0, SIGMA_FRAC * feature_std, X_raw.shape)

ALL_GROUPS = ('clinical', 'mri', 'speech')
print(f'Eval 2 (windowed, matched) dataset: {X_raw.shape} | labels {np.bincount(y)} ([Nondemented, Demented])')
print(f'Distinct composite groups: {len(set(groups))} | SIGMA_FRAC = {SIGMA_FRAC} (headline)')

Eval 2 (windowed, matched) dataset: (233, 24) | labels [105 128] ([Nondemented, Demented])
Distinct composite groups: 233 | SIGMA_FRAC = 0 (headline)


## Pipeline and per-fold metric collection

`make_pipe` builds a per-modality `StandardScaler`+`PCA(2)` fit inside the fold (no leakage),
concatenated, `MinMaxScaler((0,1))`-scaled (the `ZZFeatureMap` encodes features as rotation angles;
unscaled inputs alias past 2π and collapse the kernel), then classified. `feature_dimension =
2 x n_modalities` — 6 qubits fused, 2 qubits per single modality.

`eval_cv_full` now takes an explicit `groups` array and uses `GroupKFold` — every modality/eval in
this notebook is group-aware, not just the fused windowed case.

In [2]:
def make_pipe(kind, mod_groups, cols):
    """Per-modality StandardScaler+PCA(2) fit inside the fold, concatenated, MinMax-scaled, classified."""
    pre = ColumnTransformer([
        (g, Pipeline([('sc', StandardScaler()), ('pca', PCA(n_components=2))]), cols[g])
        for g in mod_groups
    ])
    if kind == 'svm':
        clf = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
    else:
        fmap = ZZFeatureMap(feature_dimension=2 * len(mod_groups), reps=2, entanglement='linear')
        clf = QSVC(quantum_kernel=FidelityStatevectorKernel(feature_map=fmap))
    return Pipeline([('modal', pre), ('mm', MinMaxScaler((0, 1))), ('clf', clf)])


def eval_cv_full(X, y_, groups_, kind, mod_groups, cols, n_splits=5):
    """GroupKFold(n_splits) CV, keeping predictions/scores/sex for downstream stratification.

    ROC-AUC uses decision_function rather than predict_proba: QSVC subclasses sklearn's SVC, so the
    margin is available directly, whereas probability=True would add Platt-scaling CV and stochasticity.
    """
    cv = GroupKFold(n_splits=n_splits)
    rows, cms, fold_info = [], [], []
    for tr, te in cv.split(X, y_, groups_):
        pipe = make_pipe(kind, mod_groups, cols).fit(X[tr], y_[tr])
        pred = pipe.predict(X[te])
        score = pipe.decision_function(X[te])
        cm = confusion_matrix(y_[te], pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        rows.append({
            'accuracy':    accuracy_score(y_[te], pred),
            'precision':   precision_score(y_[te], pred, zero_division=0),
            'recall':      recall_score(y_[te], pred, zero_division=0),
            'specificity': tn / (tn + fp) if (tn + fp) else 0.0,
            'f1':          f1_score(y_[te], pred, zero_division=0),
            'roc_auc':     roc_auc_score(y_[te], score) if len(set(y_[te])) > 1 else np.nan,
        })
        cms.append(cm)
        fold_info.append({'test_idx': te, 'pred': pred, 'y_true': y_[te]})
    return pd.DataFrame(rows), np.array(cms), fold_info


METRICS = ['accuracy', 'precision', 'recall', 'specificity', 'f1', 'roc_auc']


def summarise(fold_df):
    # ddof=0 (population std across the folds), matching multimodal_results.json's convention.
    return {m: (fold_df[m].mean(), fold_df[m].std(ddof=0)) for m in METRICS}


print('make_pipe / eval_cv_full ready (GroupKFold).')

make_pipe / eval_cv_full ready (GroupKFold).


## Eval 2 — per-modality + fused, on the windowed matched set

`clinical`, `mri`, `speech` alone, and `fused` (all three), each × {SVM, QSVM}, `GroupKFold(5)` on the
composite group. Headline is `SIGMA_FRAC=0`; the sensitivity table further down adds a noisy row.

In [3]:
RUNS2 = [('clinical',), ('mri',), ('speech',), ALL_GROUPS]
results2, folds2, cmats2, foldinfo2 = {}, {}, {}, {}
t_all = time.time()
for mod_groups in RUNS2:
    modality = 'fused' if mod_groups == ALL_GROUPS else mod_groups[0]
    for kind in ['svm', 'qsvm']:
        t0 = time.time()
        fdf, cms, finfo = eval_cv_full(X_raw, y, groups, kind, mod_groups, COLS)
        key = (modality, kind)
        results2[key] = summarise(fdf)
        folds2[key] = fdf
        cmats2[key] = cms
        foldinfo2[key] = finfo
        print(f'{modality:8s} {kind:4s}  acc={fdf["accuracy"].mean():.4f} +/- {fdf["accuracy"].std(ddof=0):.4f}  [{time.time()-t0:.1f}s]')
print(f'\nEval 2 total: {time.time()-t_all:.1f}s')

clinical svm   acc=0.7167 +/- 0.0260  [0.1s]


clinical qsvm  acc=0.6866 +/- 0.0509  [3.9s]
mri      svm   acc=0.7767 +/- 0.0453  [0.1s]


mri      qsvm  acc=0.7809 +/- 0.0425  [3.9s]
speech   svm   acc=0.6352 +/- 0.0143  [0.1s]


speech   qsvm  acc=0.5621 +/- 0.0621  [3.8s]
fused    svm   acc=0.8114 +/- 0.0301  [0.1s]


fused    qsvm  acc=0.7814 +/- 0.0777  [8.9s]

Eval 2 total: 20.8s


## Eval 1 — clinical + MRI only, on all ~385 OASIS subjects

No speech bottleneck: uses `data/oasis_pool.csv` directly (real labels, real clinical/MRI values,
imputation per the label-blind bake-off in `data/oasis_imputation_bakeoff.md`). `GroupKFold` on
`oasis_subject_id` — each subject is one row in the pool (last-visit representative per
`build_real_dataset.py`), so this is subject-grouped by construction.

In [4]:
pool = pd.read_csv('data/oasis_pool.csv')
y_pool = pool['label'].values
groups_pool = pool['oasis_subject_id'].values

POOL_GROUPS = {
    'clinical': ['MMSE', 'ASF', 'EDUC', 'SES'],
    'mri':      ['nWBV', 'eTIV'],
}
POOL_ALL = POOL_GROUPS['clinical'] + POOL_GROUPS['mri']
POOL_COLS = {'clinical': list(range(0, 4)), 'mri': list(range(4, 6))}
X_pool = pool[POOL_ALL].values.astype(float)

print(f'Eval 1 (OASIS pool) dataset: {X_pool.shape} | labels {np.bincount(y_pool)} ([Nondemented, Demented])')
print(f'Distinct subjects: {len(set(groups_pool))}')

RUNS1 = [('clinical',), ('mri',), ('clinical', 'mri')]
results1, folds1 = {}, {}
for mod_groups in RUNS1:
    modality = '+'.join(mod_groups)
    for kind in ['svm', 'qsvm']:
        fdf, cms, _ = eval_cv_full(X_pool, y_pool, groups_pool, kind, mod_groups, POOL_COLS)
        results1[(modality, kind)] = summarise(fdf)
        folds1[(modality, kind)] = fdf
        print(f'{modality:16s} {kind:4s}  acc={fdf["accuracy"].mean():.4f} +/- {fdf["accuracy"].std(ddof=0):.4f}')

Eval 1 (OASIS pool) dataset: (385, 6) | labels [207 178] ([Nondemented, Demented])
Distinct subjects: 385
clinical         svm   acc=0.6805 +/- 0.0650


clinical         qsvm  acc=0.6779 +/- 0.0713
mri              svm   acc=0.6156 +/- 0.0469


mri              qsvm  acc=0.6130 +/- 0.0571
clinical+mri     svm   acc=0.7429 +/- 0.0867


clinical+mri     qsvm  acc=0.7195 +/- 0.0536


## Reproduction check

Eval 2's fused accuracy (this notebook, freshly computed) must match `multimodal_qsvm.ipynb`'s
`results/multimodal_results.json` (also freshly computed, same data, same `GroupKFold(5)`, same
`SIGMA_FRAC=0`). Both notebooks build the composite group the same way (merge
`multimodal_real_provenance.csv`'s `group_id` onto the dataset by `Subject_ID`) and `GroupKFold` has
no shuffle/seed — so identical inputs must give identical folds. If this fails, the two notebooks have
drifted and neither set of numbers should be reported yet.

In [5]:
ref = json.load(open('results/multimodal_results.json'))

print('=' * 74)
print('REPRODUCTION CHECK vs results/multimodal_results.json (clean_cv, SIGMA_FRAC=0)')
print('=' * 74)
all_ok = True
for kind in ['svm', 'qsvm']:
    mean, std = results2[('fused', kind)]['accuracy']
    exp = ref['clean_cv'][kind]
    ok = (round(mean, 4) == exp['mean']) and (round(std, 4) == exp['std'])
    all_ok &= ok
    print(f'  fused {kind:4s}: got {mean:.4f} +/- {std:.4f} | published {exp["mean"]:.4f} +/- {exp["std"]:.4f} | {"MATCH" if ok else "MISMATCH"}')

print('=' * 74)
if not all_ok:
    print('MISMATCH -- multimodal_qsvm.ipynb and metrics_full.ipynb must be re-run in the same session')
    print('(same data/multimodal_dementia_dataset.csv + data/multimodal_real_provenance.csv on disk).')
assert all_ok, 'Reproduction failed -- do not report these metrics until resolved.'
print('ALL CHECKS PASSED -- metrics_full.ipynb reproduces multimodal_qsvm.ipynb exactly.')
print('=' * 74)

REPRODUCTION CHECK vs results/multimodal_results.json (clean_cv, SIGMA_FRAC=0)
  fused svm : got 0.8114 +/- 0.0301 | published 0.8114 +/- 0.0301 | MATCH
  fused qsvm: got 0.7814 +/- 0.0777 | published 0.7814 +/- 0.0777 | MATCH
ALL CHECKS PASSED -- metrics_full.ipynb reproduces multimodal_qsvm.ipynb exactly.


## Sanity checks (Phase 1.9)

- fused >= best single modality (both classifiers)
- fused not saturating (~1.0) while speech-only is near chance
- windowed fused not far above the strict-1:1 comparison build (else the delta is duplication, not signal)

In [6]:
print('--- Sanity 1: fused >= best single modality ---')
for kind in ['svm', 'qsvm']:
    fused_acc = results2[('fused', kind)]['accuracy'][0]
    singles = {m: results2[(m, kind)]['accuracy'][0] for m in ['clinical', 'mri', 'speech']}
    best_single = max(singles, key=singles.get)
    ok = fused_acc >= singles[best_single] - 1e-9
    print(f'  {kind}: fused={fused_acc:.4f}  best single={best_single} ({singles[best_single]:.4f})  '
          f'{"OK" if ok else "VIOLATION -- fused underperforms every single modality"}')

print('\n--- Sanity 2: fused not saturating while speech-only is near chance ---')
for kind in ['svm', 'qsvm']:
    fused_acc = results2[('fused', kind)]['accuracy'][0]
    speech_acc = results2[('speech', kind)]['accuracy'][0]
    print(f'  {kind}: fused={fused_acc:.4f}  speech-only={speech_acc:.4f}  '
          f'{"WATCH: fused near 1.0 while speech-only near chance (0.5)" if fused_acc > 0.97 and speech_acc < 0.6 else "OK"}')

--- Sanity 1: fused >= best single modality ---
  svm: fused=0.8114  best single=mri (0.7767)  OK
  qsvm: fused=0.7814  best single=mri (0.7809)  OK

--- Sanity 2: fused not saturating while speech-only is near chance ---
  svm: fused=0.8114  speech-only=0.6352  OK
  qsvm: fused=0.7814  speech-only=0.5621  OK


## Speech-only bootstrap CI

Speech-only GroupKFold(5) draws from only ~53 distinct speaker groups (far fewer than the ~233 rows),
so the 5 fold accuracies alone understate the real uncertainty. We percentile-bootstrap the fold
accuracies themselves (resample the 5 fold values with replacement, 5000 draws) as a cheap,
honestly-labelled approximation — a full group-level bootstrap (refitting QSVC per resample) would be
more rigorous but is expensive with the statevector simulator at this fold count.

In [7]:
rng_boot = np.random.default_rng(42)
print('Speech-only bootstrap CI (2.5th-97.5th percentile of 5000 resamples of the 5 fold accuracies):')
for kind in ['svm', 'qsvm']:
    fold_accs = folds2[('speech', kind)]['accuracy'].values
    boots = rng_boot.choice(fold_accs, size=(5000, len(fold_accs)), replace=True).mean(axis=1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    print(f'  {kind}: mean={fold_accs.mean():.4f}  95% CI [{lo:.4f}, {hi:.4f}]  (n_folds=5, n_speakers~53)')

Speech-only bootstrap CI (2.5th-97.5th percentile of 5000 resamples of the 5 fold accuracies):
  svm: mean=0.6352  95% CI [0.6205, 0.6466]  (n_folds=5, n_speakers~53)
  qsvm: mean=0.5621  95% CI [0.5193, 0.6237]  (n_folds=5, n_speakers~53)


## Sex-stratified reporting

Female subgroup is small (22 of 131 speakers overall) — disclosed here rather than hidden. Computed
from the same fused-model fold predictions already collected above (no extra fitting): within each
fold's held-out set, split by `sex` and score separately, then average across folds that had at least
one row of that sex.

In [8]:
sex_by_idx = df['sex'].values
print('Sex-stratified fused-model accuracy (from the Eval-2 fold predictions already computed):')
for kind in ['svm', 'qsvm']:
    for sex in ['F', 'M']:
        accs = []
        for info in foldinfo2[('fused', kind)]:
            te, pred, y_true = info['test_idx'], info['pred'], info['y_true']
            mask = sex_by_idx[te] == sex
            if mask.sum() == 0:
                continue
            accs.append(accuracy_score(y_true[mask], pred[mask]))
        if accs:
            print(f'  {kind:4s} sex={sex}: mean={np.mean(accs):.4f} +/- {np.std(accs, ddof=0):.4f}  '
                  f'(n_folds_with_data={len(accs)}, rows={(sex_by_idx == sex).sum()})')
        else:
            print(f'  {kind:4s} sex={sex}: no folds had test rows of this sex')

Sex-stratified fused-model accuracy (from the Eval-2 fold predictions already computed):
  svm  sex=F: mean=0.8847 +/- 0.0900  (n_folds_with_data=5, rows=106)
  svm  sex=M: mean=0.7468 +/- 0.0366  (n_folds_with_data=5, rows=127)
  qsvm sex=F: mean=0.8375 +/- 0.0824  (n_folds_with_data=5, rows=106)
  qsvm sex=M: mean=0.7317 +/- 0.0860  (n_folds_with_data=5, rows=127)


## Sensitivity table

Axes actually re-run this pass: matching richness (`R_max` 10 / 18(default) / 25, plus the strict
1:1 comparison build) and raw-vs-clean speech, each on the fused model; noise on/off (`SIGMA_FRAC` 0
vs 1.5) on the default windowed/raw build. All GroupKFold(5) on the composite group, SVM + QSVM.

**Not re-run this pass** (disclosed, not silently dropped): the OASIS imputation scheme (S0-S4) was
selected once by label-blind reconstruction error in `data/oasis_imputation_bakeoff.md` (S2 won); a
full per-scheme downstream-CV re-run was not repeated here. The age-match `tol` (8/12/15) is fixed at
`speakers_meta.csv` merge time (Phase 1.5), not a matcher CLI parameter, so a `tol` sweep was not
re-run either. Both are noted as deferred sensitivity axes, not resolved ones.

In [9]:
def load_variant(path_csv, path_prov):
    d = pd.read_csv(path_csv)
    p = pd.read_csv(path_prov)
    d = d.merge(p[['Subject_ID', 'group_id']], on='Subject_ID', how='left')
    assert d['group_id'].isna().sum() == 0, f'{path_csv} rows missing a matching group_id in {path_prov}'
    Xv = d[ALL_FEATURES].values.astype(float)
    yv = d['Label'].values
    gv = d['group_id'].values
    return Xv, yv, gv


def fused_acc(Xv, yv, gv, sigma=0.0):
    fstd = Xv.std(axis=0)
    Xn = Xv + np.random.default_rng(42).normal(0, sigma * fstd, Xv.shape) if sigma else Xv
    out = {}
    for kind in ['svm', 'qsvm']:
        fdf, _, _ = eval_cv_full(Xn, yv, gv, kind, ALL_GROUPS, COLS)
        out[kind] = (fdf['accuracy'].mean(), fdf['accuracy'].std(ddof=0))
    return out


sensitivity_rows = []

# R_max variants + strict 1:1, raw speech
variants = [
    ('R_max=10',   'data/sensitivity/r10/multimodal_dementia_dataset_raw.csv',
                   'data/sensitivity/r10/multimodal_real_provenance_raw.csv'),
    ('R_max=18 (default)', 'data/multimodal_dementia_dataset_raw.csv',
                   'data/multimodal_real_provenance_raw.csv'),
    ('R_max=25',   'data/sensitivity/r25/multimodal_dementia_dataset_raw.csv',
                   'data/sensitivity/r25/multimodal_real_provenance_raw.csv'),
    ('strict 1:1', 'data/sensitivity/strict1to1/multimodal_dementia_dataset_raw.csv',
                   'data/sensitivity/strict1to1/multimodal_real_provenance_raw.csv'),
]
for label, csvp, provp in variants:
    Xv, yv, gv = load_variant(csvp, provp)
    accs = fused_acc(Xv, yv, gv, sigma=0.0)
    for kind, (m, s) in accs.items():
        sensitivity_rows.append({'axis': 'matching richness', 'variant': label, 'n_rows': len(yv),
                                  'model': kind, 'acc_mean': round(float(m), 4), 'acc_std': round(float(s), 4)})

# raw vs clean speech, default R_max
for label, csvp, provp in [
    ('raw',   'data/multimodal_dementia_dataset_raw.csv',   'data/multimodal_real_provenance_raw.csv'),
    ('clean', 'data/multimodal_dementia_dataset_clean.csv', 'data/multimodal_real_provenance_clean.csv'),
]:
    Xv, yv, gv = load_variant(csvp, provp)
    accs = fused_acc(Xv, yv, gv, sigma=0.0)
    for kind, (m, s) in accs.items():
        sensitivity_rows.append({'axis': 'speech source', 'variant': label, 'n_rows': len(yv),
                                  'model': kind, 'acc_mean': round(float(m), 4), 'acc_std': round(float(s), 4)})

# noise on/off, default raw windowed
for sigma, label in [(0.0, 'SIGMA_FRAC=0 (off, headline)'), (1.5, 'SIGMA_FRAC=1.5 (on)')]:
    accs = fused_acc(X_raw, y, groups, sigma=sigma)
    for kind, (m, s) in accs.items():
        sensitivity_rows.append({'axis': 'noise injection', 'variant': label, 'n_rows': len(y),
                                  'model': kind, 'acc_mean': round(float(m), 4), 'acc_std': round(float(s), 4)})

sens_table = pd.DataFrame(sensitivity_rows)
pd.set_option('display.width', 200)
print(sens_table.to_string(index=False))

# Sanity 3: windowed fused not far above strict-1:1 fused
w = sens_table[(sens_table.axis == 'matching richness') & (sens_table.variant == 'R_max=18 (default)')]
s = sens_table[(sens_table.axis == 'matching richness') & (sens_table.variant == 'strict 1:1')]
print('\n--- Sanity 3: windowed fused vs strict-1:1 fused (large gap would mean duplication, not signal) ---')
for kind in ['svm', 'qsvm']:
    wm = w[w.model == kind]['acc_mean'].iloc[0]
    sm = s[s.model == kind]['acc_mean'].iloc[0]
    gap = wm - sm
    print(f'  {kind}: windowed={wm:.4f}  strict-1:1={sm:.4f}  gap={gap:+.4f}  '
          f'{"WATCH: large positive gap" if gap > 0.1 else "OK"}')

             axis                      variant  n_rows model  acc_mean  acc_std
matching richness                     R_max=10     221   svm    0.8505   0.0374
matching richness                     R_max=10     221  qsvm    0.7734   0.0487
matching richness           R_max=18 (default)     233   svm    0.8114   0.0301
matching richness           R_max=18 (default)     233  qsvm    0.7814   0.0777
matching richness                     R_max=25     233   svm    0.8114   0.0301
matching richness                     R_max=25     233  qsvm    0.7814   0.0777
matching richness                   strict 1:1     101   svm    0.7633   0.0686
matching richness                   strict 1:1     101  qsvm    0.7638   0.0908
    speech source                          raw     233   svm    0.8114   0.0301
    speech source                          raw     233  qsvm    0.7814   0.0777
    speech source                        clean     233   svm    0.8155   0.0623
    speech source                       

## Results table

In [10]:
LABELS = {'svm': 'Classical SVM', 'qsvm': 'QSVM (ZZFeatureMap)'}
MODNAME2 = {'clinical': 'Clinical only (4 feat)', 'mri': 'MRI only (2 feat)',
            'speech': 'Speech only (18 feat)', 'fused': 'Multimodal fused (24 feat)'}

rows = []
for (modality, kind), summ in results2.items():
    r = {'eval_name': 'Eval 2 (windowed, N=%d)' % len(y), 'modality': MODNAME2[modality], 'model': LABELS[kind],
         'n_qubits': 6 if modality == 'fused' else 2}
    for m in METRICS:
        r[f'{m}_mean'], r[f'{m}_std'] = round(summ[m][0], 4), round(summ[m][1], 4)
    rows.append(r)
for (modality, kind), summ in results1.items():
    r = {'eval_name': 'Eval 1 (OASIS pool, N=%d)' % len(y_pool), 'modality': modality, 'model': LABELS[kind],
         'n_qubits': 2 * len(modality.split('+'))}
    for m in METRICS:
        r[f'{m}_mean'], r[f'{m}_std'] = round(summ[m][0], 4), round(summ[m][1], 4)
    rows.append(r)
table = pd.DataFrame(rows)

pd.set_option('display.width', 200, 'display.max_columns', 40)
print('--- Eval 2 (windowed, fused + per-modality, SIGMA_FRAC=0) ---')
print(table[table['eval_name'].str.startswith('Eval 2')].drop(columns='eval_name').to_string(index=False))
print('\n--- Eval 1 (clinical/MRI only, all ~385 OASIS subjects) ---')
print(table[table['eval_name'].str.startswith('Eval 1')].drop(columns='eval_name').to_string(index=False))

--- Eval 2 (windowed, fused + per-modality, SIGMA_FRAC=0) ---
                  modality               model  n_qubits  accuracy_mean  accuracy_std  precision_mean  precision_std  recall_mean  recall_std  specificity_mean  specificity_std  f1_mean  f1_std  roc_auc_mean  roc_auc_std
    Clinical only (4 feat)       Classical SVM         2         0.7167        0.0260          0.7526         0.0458       0.7270      0.0522            0.7052           0.0748   0.7373  0.0270        0.7734       0.0281
    Clinical only (4 feat) QSVM (ZZFeatureMap)         2         0.6866        0.0509          0.7273         0.1090       0.7644      0.1527            0.6040           0.2132   0.7223  0.0574        0.7611       0.0429
         MRI only (2 feat)       Classical SVM         2         0.7767        0.0453          0.7544         0.0475       0.8815      0.0460            0.6458           0.0800   0.8122  0.0395        0.8434       0.0393
         MRI only (2 feat) QSVM (ZZFeatureMap)        

### Paste-ready markdown table

Percentages as `mean ± std` over the 5 folds.

In [11]:
def md_table(sub, caption):
    head = '| Model | Accuracy | Precision | Recall (Sens.) | Specificity | F1 | ROC-AUC |'
    sep  = '|---|---|---|---|---|---|---|'
    lines = [f'**{caption}**', '', head, sep]
    for _, r in sub.iterrows():
        name = f"{r['modality']} -- {r['model']}"
        cells_ = [f"{r[f'{m}_mean']*100:.2f} +/- {r[f'{m}_std']*100:.2f}" for m in METRICS]
        lines.append('| ' + name + ' | ' + ' | '.join(cells_) + ' |')
    return '\n'.join(lines)

order2 = ['Clinical only (4 feat)', 'MRI only (2 feat)', 'Speech only (18 feat)', 'Multimodal fused (24 feat)']
e2 = table[table['eval_name'].str.startswith('Eval 2')].copy()
e2['_o'] = e2['modality'].map({k: i for i, k in enumerate(order2)})
e2 = e2.sort_values(['_o', 'model']).drop(columns='_o')

print(md_table(e2, f'Table -- Eval 2, 5-fold GroupKFold CV, N={len(y)}, SIGMA_FRAC=0, positive class = Demented'))
print()
print(md_table(table[table['eval_name'].str.startswith('Eval 1')],
               f'Table -- Eval 1, 5-fold GroupKFold CV, N={len(y_pool)} (all OASIS subjects), clinical/MRI only'))

**Table -- Eval 2, 5-fold GroupKFold CV, N=233, SIGMA_FRAC=0, positive class = Demented**

| Model | Accuracy | Precision | Recall (Sens.) | Specificity | F1 | ROC-AUC |
|---|---|---|---|---|---|---|
| Clinical only (4 feat) -- Classical SVM | 71.67 +/- 2.60 | 75.26 +/- 4.58 | 72.70 +/- 5.22 | 70.52 +/- 7.48 | 73.73 +/- 2.70 | 77.34 +/- 2.81 |
| Clinical only (4 feat) -- QSVM (ZZFeatureMap) | 68.66 +/- 5.09 | 72.73 +/- 10.90 | 76.44 +/- 15.27 | 60.40 +/- 21.32 | 72.23 +/- 5.74 | 76.11 +/- 4.29 |
| MRI only (2 feat) -- Classical SVM | 77.67 +/- 4.53 | 75.44 +/- 4.75 | 88.15 +/- 4.60 | 64.58 +/- 8.00 | 81.22 +/- 3.95 | 84.34 +/- 3.93 |
| MRI only (2 feat) -- QSVM (ZZFeatureMap) | 78.09 +/- 4.25 | 79.18 +/- 6.82 | 82.87 +/- 4.97 | 72.23 +/- 10.40 | 80.66 +/- 3.15 | 82.82 +/- 2.75 |
| Speech only (18 feat) -- Classical SVM | 63.52 +/- 1.43 | 64.23 +/- 2.46 | 75.68 +/- 7.26 | 48.56 +/- 6.04 | 69.30 +/- 3.43 | 71.12 +/- 3.70 |
| Speech only (18 feat) -- QSVM (ZZFeatureMap) | 56.21 +/- 6.21 |

## Save artifacts

In [12]:
out = {
    'source_notebook': 'metrics_full.ipynb',
    'description': 'Full metric set, real data, two evaluations (windowed fused + OASIS-pool clinical/MRI)',
    'eval2_dataset': 'data/multimodal_dementia_dataset.csv (real, windowed match)',
    'eval2_n': int(len(y)),
    'eval1_dataset': 'data/oasis_pool.csv (all labelled OASIS subjects)',
    'eval1_n': int(len(y_pool)),
    'positive_class': 'Demented (Label == 1)',
    'cv': 'GroupKFold(n_splits=5) on the composite group (Eval 2) / oasis_subject_id (Eval 1)',
    'std_convention': 'population std across the 5 folds (ddof=0)',
    'roc_auc_source': 'decision_function (not predict_proba)',
    'sigma_frac_headline': SIGMA_FRAC,
    'reproduction_check': 'PASSED -- Eval 2 fused accuracy matches results/multimodal_results.json (clean_cv)',
    'sensitivity_table': sensitivity_rows,
    'eval2_runs': {}, 'eval1_runs': {},
}
for (modality, kind), summ in results2.items():
    out['eval2_runs'][f'{modality}|{kind}'] = {
        'modality': modality, 'model': kind,
        'n_qubits': 6 if modality == 'fused' else 2,
        'metrics': {m: {'mean': round(summ[m][0], 4), 'std': round(summ[m][1], 4)} for m in METRICS},
        'per_fold': {m: [round(v, 4) if not np.isnan(v) else None for v in folds2[(modality, kind)][m].tolist()] for m in METRICS},
        'confusion_matrices_per_fold': cmats2[(modality, kind)].tolist(),
    }
for (modality, kind), summ in results1.items():
    out['eval1_runs'][f'{modality}|{kind}'] = {
        'modality': modality, 'model': kind,
        'metrics': {m: {'mean': round(summ[m][0], 4), 'std': round(summ[m][1], 4)} for m in METRICS},
        'per_fold': {m: [round(v, 4) if not np.isnan(v) else None for v in folds1[(modality, kind)][m].tolist()] for m in METRICS},
    }

with open('results/metrics_full.json', 'w') as f:
    json.dump(out, f, indent=2)
table.to_csv('results/metrics_table.csv', index=False)
print('Saved: results/metrics_full.json')
print('Saved: results/metrics_table.csv')

Saved: results/metrics_full.json
Saved: results/metrics_table.csv


## Notes for the write-up

- **Two honest evaluations, not one inflated one.** Eval 1 (clinical+MRI, all ~385 OASIS subjects)
  shows what the tabular data alone can do at full sample size; Eval 2 (fused, windowed, 233 rows)
  shows the actual multimodal claim, honestly scoped to the ~53-131 real voices behind it.
- **No leak, no synthetic-style saturation.** Single-feature stumps on MMSE/nWBV land around 0.7-0.85
  (see `scripts/verify_real_dataset.py`), not the synthetic dataset's ~0.996 `hippocampal_volume`
  stump — `SIGMA_FRAC=0` is honest here in a way it wasn't on the old synthetic set.
- **Speech-only is the noisiest branch** — only ~53 distinct speakers feed it under the default
  `R_max`; the bootstrap CI above is the honest disclosure of that, not a hidden caveat.
- **Female subgroup is small** (22 of 131 speakers total) — reported above per fold rather than
  suppressed; treat with proportionate caution.
- **All figures on real data, cross-matched by (label, sex, age), not measured jointly on the same
  people.** See `validation_checks.ipynb` for the representativeness diagnostic and
  `data/DATASHEET.md` for the full limitations list.